In [65]:
EMA_path = "./../data/datasets/final_1995/EMA.csv"
JAPAN_path = "./../data/datasets/final_1995/PMDA.csv"
SWISSMEDIC_path = "./../data/datasets/final_1995/Swissmedic.csv"
AUSTRALIA_path = "./../data/datasets/final_1995/TGA.csv"
FDA_path = "./../data/datasets/final_1995/FDA.csv"
HEALTHCANADA_path = "./../data/datasets/final_1995/HealthCanada.csv"


In [66]:
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
from matplotlib import cm
import matplotlib.colors as mcolors
import seaborn as sns
import json
import numpy as np
from matplotlib.ticker import FixedLocator, ScalarFormatter
import math
import os

In [68]:
# =============================
# CSV Loader
# =============================
def load_agency_csv(path: str, agency: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    # Spaltennamen bereinigen (falls irgendwo Leerzeichen sind)
    df.columns = df.columns.astype(str).str.strip()
    return df

# =============================
# Load all agencies into dfs
# =============================
df_ema = load_agency_csv(EMA_path, "EMA")
df_fda = load_agency_csv(FDA_path, "FDA")
df_swissmedic = load_agency_csv(SWISSMEDIC_path, "SWISSMEDIC")
df_japan = load_agency_csv(JAPAN_path, "JAPAN")
df_australia = load_agency_csv(AUSTRALIA_path, "AUSTRALIA")
df_healthcanada = load_agency_csv(HEALTHCANADA_path, "HEALTHCANADA")

# =============================
# Helper für Identifier-Zählung
# =============================
PLACEHOLDERS = {"not reported", "na", "n/a", "tbd", "none", ""}

def cleaned_series(s: pd.Series) -> pd.Series:
    x = s.astype("string").str.strip()
    x = x.mask(x.str.lower().isin(PLACEHOLDERS))
    return x

def agency_identifier_count(df: pd.DataFrame, agency: str) -> int:
    """
    Zählt den passenden Identifier pro Agency:
    - Japan: Anzahl Zeilen (= PDFs/Records in deinem CSV)
    - alle anderen: unique Marketing_authorisation_number (ohne Platzhalter)
    """
    if agency == "JAPAN":
        return len(df)

    s = cleaned_series(df["Marketing_authorisation_number"])
    return s.nunique(dropna=True)

# Results 1. Dataset Characteristics

Numbers per application

In [69]:
agencies = {
    "FDA": df_fda,
    "Health Canada": df_healthcanada,
    "EMA": df_ema,
    "Swissmedic": df_swissmedic,
    "Japan": df_japan,
    "Australia": df_australia,
}

# ============================================================
# 1) Record counts per agency (based on CSV rows)
# ============================================================
rows = []
total_records = sum(len(df) for df in agencies.values())

for name, df in agencies.items():
    n = len(df)
    pct = round(n / total_records * 100, 2) if total_records else 0.0
    rows.append({
        "Agency": name,
        "n_records": n,
        "%_of_overall_records": pct
    })

record_summary = pd.DataFrame(rows)

print("Record counts per agency (based on CSV rows)")
display(record_summary)

Record counts per agency (based on CSV rows)


,Agency,n_records,%_of_overall_records
0,FDA,18558,58.06
1,Health Canada,10286,32.18
2,EMA,1491,4.66
3,Swissmedic,233,0.73
4,Japan,408,1.28
5,Australia,988,3.09


#Number per Approval (Consolidated from Approved, Conditional Marketing Authorisation, Marketed)

In [72]:
# ============================================================
# Approved applications per agency (RECORD COUNTS, no normalisation)
# ============================================================

# Decisions, die als Approval zählen (exakt so geschrieben)
APPROVAL_DECISIONS = {
    "approved",
    "conditional marketing authorisation",
    "marketed",
}

rows = []
total_records_approved = 0

for name, df in agencies.items():
    # Anzahl Records insgesamt
    n_records = len(df)

    # Anzahl Records mit Approval-Decision
    n_approved = int(df["Decision"].isin(APPROVAL_DECISIONS).sum())

    rows.append({
        "Agency": name,
        "n_records_approved": n_approved,
    })

    total_records_approved += n_approved

approved_record_summary = pd.DataFrame(rows)

# Prozentanteil über alle Agencies (analog zu deinen anderen Tabellen)
approved_record_summary["%_of_overall_approved_records"] = (
    approved_record_summary["n_records_approved"] / total_records_approved * 100
).round(2)

print("Approved applications per agency (record counts, no normalisation)")
display(approved_record_summary)


Approved applications per agency (record counts, no normalisation)


,Agency,n_records_approved,%_of_overall_approved_records
0,FDA,18558,58.06
1,Health Canada,10286,32.18
2,EMA,1491,4.66
3,Swissmedic,233,0.73
4,Japan,408,1.28
5,Australia,988,3.09


Numbers per drug (unique Marketing_authorisation_number)

In [70]:
# ============================================================
# 2) Unique identifiers per agency (MA numbers; Japan = rows)
# ============================================================
def clean_ma_series(s: pd.Series) -> set:
    return set(
        s.astype("string")
         .str.strip()
         .str.lower()
         .mask(lambda x: x.isin(PLACEHOLDERS))
         .dropna()
         .unique()
    )

agency_ids = {}
overall_ids = 0

for name, df in agencies.items():
    if name == "Japan":
        n_ids = len(df)
        agency_ids[name] = n_ids
        overall_ids += n_ids
        continue

    ma_set = clean_ma_series(df["Marketing_authorisation_number"])
    n_ids = len(ma_set)
    agency_ids[name] = n_ids
    overall_ids += n_ids

rows = []
for name, n in agency_ids.items():
    pct = round(n / overall_ids * 100, 2) if overall_ids else 0.0
    rows.append({
        "Agency": name,
        "n_unique_identifiers": n,
        "%_of_overall": pct
    })

ma_summary = pd.DataFrame(rows)

print("Unique identifiers per agency (MA numbers; Japan = rows)")
display(ma_summary)

Unique identifiers per agency (MA numbers; Japan = rows)


,Agency,n_unique_identifiers,%_of_overall
0,FDA,18558,58.69
1,Health Canada,10286,32.53
2,EMA,1305,4.13
3,Swissmedic,195,0.62
4,Japan,408,1.29
5,Australia,871,2.75


# Table 1: Key characteristics of approvals

In [71]:
# ============================================================
# Constants & helpers (analysis)
# ============================================================
DRUG_CLASS_ORDER = [
    "small molecule",
    "biologics",
    "cell and gene therapy",
    "peptides and proteins",
    "vaccine",
    "not reported",
    "other",
]

def norm_series(s: pd.Series) -> pd.Series:
    """Strip whitespace, lower-case, keep NaN."""
    return s.astype("string").str.strip().str.lower()

# ============================================================
# Decision distribution (record-level, per agency)
# Output wie in deinem Screenshot (inkl. consolidated)
# ============================================================
def decision_distribution(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)

    decision = (
        df.get("Decision", pd.Series([pd.NA] * n))
          .astype("string")
          .str.strip()
          .str.lower()
    )

    # Platzhalter -> NA (damit <NA> separat erscheint)
    decision = decision.mask(decision.isin(PLACEHOLDERS), pd.NA)

    counts = decision.value_counts(dropna=False)
    dist = counts.reset_index()
    dist.columns = ["Decision", "n"]
    dist["%"] = (dist["n"] / n * 100).round(2) if n else 0.0

    consolidated_mask = decision.isin({
        "approved",
        "conditional marketing authorisation",
        "conditional marketing authorization",
    })

    consolidated_row = pd.DataFrame([{
        "Decision": "consolidated (approved + conditional marketing authorisation)",
        "n": int(consolidated_mask.sum()),
        "%": round(consolidated_mask.sum() / n * 100, 2) if n else 0.0
    }])

    return pd.concat([dist, consolidated_row], ignore_index=True)

# ============================================================
# Drug class summary (record-level)
# ============================================================
def bucket_drug_class(x) -> str:
    if pd.isna(x):
        return "not reported"

    t = str(x).strip().lower()
    if t in PLACEHOLDERS:
        return "not reported"

    if t in DRUG_CLASS_ORDER:
        return t

    return "other"

def drug_class_summary(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)
    s = df.get("Drug_class", pd.Series([pd.NA] * n)).map(bucket_drug_class)
    counts = s.value_counts()

    rows = []
    for c in DRUG_CLASS_ORDER:
        k = int(counts.get(c, 0))
        rows.append({
            "Drug class": c,
            "n": k,
            "%": round(k / n * 100, 2) if n else 0.0
        })
    return pd.DataFrame(rows)

# ============================================================
# Therapeutic area composition (Top 5, mention-based)
# ============================================================
def therapeutic_area_top5(df: pd.DataFrame) -> pd.DataFrame:
    col = "Disease_class(es)"
    if col not in df.columns:
        return pd.DataFrame(columns=["Therapeutic area", "n", "%"])

    s = norm_series(df[col]).mask(lambda x: x.isin(PLACEHOLDERS)).dropna()
    if s.empty:
        return pd.DataFrame(columns=["Therapeutic area", "n", "%"])

    exploded = (
        s.str.split(";")
         .explode()
         .astype("string")
         .str.strip()
         .str.lower()
    )
    exploded = exploded[~exploded.isin(PLACEHOLDERS)].dropna()

    counts = exploded.value_counts().head(5)
    denom = int(exploded.shape[0])

    out = counts.reset_index()
    out.columns = ["Therapeutic area", "n"]
    out["%"] = (out["n"] / denom * 100).round(2) if denom else 0.0
    return out

FOCUS_DISEASE_CLASSES = [
    "diseases of the circulatory system",
    "diseases of the nervous system",
    "neoplasms",
    "endocrine, nutritional and metabolic diseases",
    "infectious diseases",
]

def therapeutic_area_focus5(df: pd.DataFrame) -> pd.DataFrame:
    col = "Disease_class(es)"
    if col not in df.columns:
        return pd.DataFrame(columns=["Therapeutic area", "n", "%"])

    s = (
        df[col]
        .astype("string")
        .str.strip()
        .str.lower()
        .mask(lambda x: x.isin(PLACEHOLDERS))
        .dropna()
    )
    if s.empty:
        return pd.DataFrame(
            [{"Therapeutic area": c, "n": 0, "%": 0.0} for c in FOCUS_DISEASE_CLASSES]
        )

    exploded = (
        s.str.split(";")
         .explode()
         .astype("string")
         .str.strip()
         .str.lower()
    )
    exploded = exploded[~exploded.isin(PLACEHOLDERS)].dropna()

    denom = int(exploded.shape[0])  # mention-based (wie Top 5)
    counts = exploded.value_counts()

    rows = []
    for c in FOCUS_DISEASE_CLASSES:
        n = int(counts.get(c, 0))
        pct = round(n / denom * 100, 2) if denom else 0.0
        rows.append({"Therapeutic area": c, "n": n, "%": pct})

    return pd.DataFrame(rows)


# ============================================================
# Run for all agencies
# ============================================================
for name, df in agencies.items():
    print("\n" + "=" * 70)
    print(f"{name} | Records: {len(df)}")

    print("\nDecision distribution (per application)")
    display(decision_distribution(df))

    print("\nDrug classes (applications)")
    display(drug_class_summary(df))

    print("\nTherapeutic area composition (Top 5)")
    display(therapeutic_area_top5(df))

# ============================================================
# OVERALL
# ============================================================
df_overall = pd.concat(list(agencies.values()), ignore_index=True)

print("\n" + "=" * 70)
print("OVERALL (across all agencies)")
print(f"Records: {len(df_overall)}")

print("\nDecision distribution (OVERALL)")
display(decision_distribution(df_overall))

print("\nDrug classes (OVERALL)")
display(drug_class_summary(df_overall))

print("\nTherapeutic area composition (Top 5, OVERALL)")
display(therapeutic_area_top5(df_overall))

print("\nTherapeutic area composition (selected 5 disease classes)")
display(therapeutic_area_focus5(df))



FDA | Records: 18558

Decision distribution (per application)


,Decision,n,%
0,approved,17502,94.31
1,conditional marketing authorisation,1056,5.69
2,consolidated (approved + conditional marketing...,18558,100.0



Drug classes (applications)


,Drug class,n,%
0,small molecule,16881,90.96
1,biologics,335,1.81
2,cell and gene therapy,0,0.00
3,peptides and proteins,604,3.25
4,vaccine,0,0.00
5,not reported,346,1.86
6,other,392,2.11



Therapeutic area composition (Top 5)


...

,Drug class,n,%
0,small molecule,27480,85.97
1,biologics,1696,5.31
2,cell and gene therapy,47,0.15
3,peptides and proteins,1258,3.94
4,vaccine,307,0.96
5,not reported,346,1.08
6,other,830,2.60



Therapeutic area composition (Top 5, OVERALL)


,Therapeutic area,n,%
0,diseases of the circulatory system,2673,10.98
1,diseases of the nervous system,2304,9.46
2,neoplasms,2268,9.32
3,"endocrine, nutritional, and metabolic diseases",2143,8.8
4,infectious and parasitic diseases,2070,8.5



Therapeutic area composition (selected 5 disease classes)


,Therapeutic area,n,%
0,diseases of the circulatory system,81,5.82
1,diseases of the nervous system,80,5.75
2,neoplasms,268,19.25
3,"endocrine, nutritional and metabolic diseases",0,0.00
4,infectious diseases,0,0.00


# 4. Administration routes and pharmaceutical forms

In [88]:
def administration_route_top10(df: pd.DataFrame) -> pd.DataFrame:
    """
    Top 10 Administration routes (mention-based).
    Parenteral routes (intravenous, subcutaneous, intramuscular)
    are aggregated into a single category: 'parenteral'.
    """
    col = "Administration_route"
    if col not in df.columns:
        return pd.DataFrame(columns=["Administration route", "n", "%"])

    s = (
        df[col]
        .astype("string")
        .str.strip()
        .str.lower()
        .mask(lambda x: x.isin(PLACEHOLDERS))
        .dropna()
    )

    if s.empty:
        return pd.DataFrame(columns=["Administration route", "n", "%"])

    exploded = (
        s.str.split(";")
         .explode()
         .astype("string")
         .str.strip()
    )
    exploded = exploded[~exploded.isin(PLACEHOLDERS)].dropna()

    # ---- aggregate parenteral routes
    exploded = exploded.replace({
        "intravenous": "parenteral",
        "subcutaneous": "parenteral",
        "intramuscular": "parenteral",
    })

    counts = exploded.value_counts().head(10)
    denom = int(exploded.shape[0])

    out = counts.reset_index()
    out.columns = ["Administration route", "n"]
    out["%"] = (out["n"] / denom * 100).round(2) if denom else 0.0
    return out


def pharmaceutical_form_top10(df: pd.DataFrame) -> pd.DataFrame:
    """
    Top 10 Pharmaceutical forms (mention-based).
    Aggregations:
      - tablet + capsule -> 'tablet / capsule'
      - solution + injectable + suspension -> 'solution / injectable'
    """
    col = "Pharmaceutical_form"
    if col not in df.columns:
        return pd.DataFrame(columns=["Pharmaceutical form", "n", "%"])

    s = (
        df[col]
        .astype("string")
        .str.strip()
        .str.lower()
        .mask(lambda x: x.isin(PLACEHOLDERS))
        .dropna()
    )

    if s.empty:
        return pd.DataFrame(columns=["Pharmaceutical form", "n", "%"])

    exploded = (
        s.str.split(";")
         .explode()
         .astype("string")
         .str.strip()
    )
    exploded = exploded[~exploded.isin(PLACEHOLDERS)].dropna()

    # ---- aggregate pharmaceutical forms
    exploded = exploded.replace({
        "tablet": "tablet / capsule",
        "capsule": "tablet / capsule",
        "solution": "solution / injectable",
        "injectable": "solution / injectable",
    })

    counts = exploded.value_counts().head(10)
    denom = int(exploded.shape[0])

    out = counts.reset_index()
    out.columns = ["Pharmaceutical form", "n"]
    out["%"] = (out["n"] / denom * 100).round(2) if denom else 0.0
    return out

for name, df in agencies.items():
    print("\n" + "=" * 70)
    print(f"{name}")

    print("\nAdministration route (Top 5)")
    display(administration_route_top10(df))

    print("\nPharmaceutical form (Top 5)")
    display(pharmaceutical_form_top10(df))


print("\n" + "=" * 70)
print(f"Overall")

display(administration_route_top10(df_overall))
display(pharmaceutical_form_top10(df_overall))



FDA

Administration route (Top 5)


,Administration route,n,%
0,oral,17469,61.51
1,injection,4946,17.42
2,parenteral,1901,6.69
3,topical,1710,6.02
4,ophthalmic,779,2.74
5,inhalation,471,1.66
6,nasal,173,0.61
7,transdermal,155,0.55
8,vaginal,143,0.5
9,sublingual,118,0.42



Pharmaceutical form (Top 5)


,Pharmaceutical form,n,%
0,tablet / capsule,15552,53.5
1,solution / injectable,8298,28.54
2,suspension,876,3.01
3,drops,707,2.43
4,cream,609,2.09
5,ointment,428,1.47
6,powder,414,1.42
7,syrup,302,1.04
8,gel,268,0.92
9,spray,185,0.64



Health Canada

Administration route (Top 5)


...


Pharmaceutical form (Top 5)


,Pharmaceutical form,n,%
0,tablet / capsule,424,37.82
1,solution / injectable,296,26.4
2,powder,159,14.18
3,suspension,79,7.05
4,concentrate,58,5.17
5,lyophilized powder,11,0.98
6,solvent,10,0.89
7,injection,9,0.8
8,diluent,8,0.71
9,spray,7,0.62



Overall


,Administration route,n,%
0,oral,26465,59.49
1,parenteral,6262,14.08
2,injection,4951,11.13
3,topical,2834,6.37
4,ophthalmic,997,2.24
5,inhalation,741,1.67
6,nasal,256,0.58
7,transdermal,226,0.51
8,vaginal,193,0.43
9,sublingual,183,0.41


,Pharmaceutical form,n,%
0,tablet / capsule,24114,53.92
1,solution / injectable,11022,24.65
2,powder,1705,3.81
3,suspension,1261,2.82
4,cream,901,2.01
5,drops,757,1.69
6,ointment,527,1.18
7,lotion,451,1.01
8,gel,399,0.89
9,syrup,392,0.88


# 6. Regulatory review durations

In [91]:
import pandas as pd
from IPython.display import display

# ============================================================
# Helper: compute review duration (POSITIVE ONLY)
# ============================================================
def compute_review_duration(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds 'review_duration_days' = Decision_date - Application_date.
    Keeps ONLY positive durations (> 0 days).
    """
    tmp = df.copy()

    tmp["Application_date"] = pd.to_datetime(
        tmp["Application_date"], errors="coerce", dayfirst=True
    )
    tmp["Decision_date"] = pd.to_datetime(
        tmp["Decision_date"], errors="coerce", dayfirst=True
    )

    tmp["review_duration_days"] = (
        tmp["Decision_date"] - tmp["Application_date"]
    ).dt.days

    # keep only positive durations
    tmp = tmp[tmp["review_duration_days"] > 0]

    return tmp.dropna(subset=["review_duration_days"])


# ============================================================
# Median, Q1, Q3, IQR (positive durations only)
# ============================================================
def review_duration_median_iqr(df: pd.DataFrame) -> pd.DataFrame:
    """
    Median, Q1, Q3 and IQR (Q3 - Q1) of positive review durations (days),
    stratified by abridged vs non-abridged.
    """
    tmp = compute_review_duration(df)

    tmp["Procedure"] = (
        tmp["Nonclinical_abridged"]
        .astype("string")
        .str.strip()
        .str.lower()
        .map({"yes": "Abridged", "no": "Non-abridged"})
    )

    tmp = tmp.dropna(subset=["Procedure"])

    out = (
        tmp.groupby("Procedure")["review_duration_days"]
        .agg(
            median_days="median",
            q1_days=lambda x: x.quantile(0.25),
            q3_days=lambda x: x.quantile(0.75),
        )
        .reset_index()
    )

    out["iqr_days"] = out["q3_days"] - out["q1_days"]

    return out


# ============================================================
# Min / Max (positive durations only)
# ============================================================
def review_duration_min_max(df: pd.DataFrame) -> pd.DataFrame:
    """
    Min and max positive review duration (days),
    stratified by abridged vs non-abridged.
    """
    tmp = compute_review_duration(df)

    tmp["Procedure"] = (
        tmp["Nonclinical_abridged"]
        .astype("string")
        .str.strip()
        .str.lower()
        .map({"yes": "Abridged", "no": "Non-abridged"})
    )

    tmp = tmp.dropna(subset=["Procedure"])

    out = (
        tmp.groupby("Procedure")["review_duration_days"]
        .agg(
            min_days="min",
            max_days="max",
        )
        .reset_index()
    )

    return out


# ============================================================
# Agencies to include (FDA & Health Canada excluded)
# ============================================================
agencies_timeline = {
    "EMA": df_ema,
    "Swissmedic": df_swissmedic,
    "PMDA": df_japan,
    "TGA": df_australia,
}

# ============================================================
# Per-agency outputs
# ============================================================
for name, df in agencies_timeline.items():
    print("\n" + "=" * 70)
    print(f"{name} – Review timelines (positive durations only)")

    print("\nMedian, Q1, Q3 and IQR (days)")
    display(review_duration_median_iqr(df))

    print("\nMin / Max (days)")
    display(review_duration_min_max(df))


# ============================================================
# OVERALL (across included agencies)
# ============================================================
df_overall_timeline = pd.concat(list(agencies_timeline.values()), ignore_index=True)

print("\n" + "=" * 70)
print("OVERALL – Review timelines (positive durations only)")

print("\nMedian, Q1, Q3 and IQR (days)")
display(review_duration_median_iqr(df_overall_timeline))

print("\nMin / Max (days)")
display(review_duration_min_max(df_overall_timeline))



EMA – Review timelines (positive durations only)

Median, Q1, Q3 and IQR (days)


,Procedure,median_days,q1_days,q3_days,iqr_days
0,Abridged,331.0,254.5,409.0,154.5
1,Non-abridged,384.0,335.0,476.0,141.0



Min / Max (days)


,Procedure,min_days,max_days
0,Abridged,9.0,1004.0
1,Non-abridged,2.0,2090.0



Swissmedic – Review timelines (positive durations only)

Median, Q1, Q3 and IQR (days)


...

,Procedure,min_days,max_days
0,Abridged,4.0,2388.0
1,Non-abridged,32.0,3284.0



OVERALL – Review timelines (positive durations only)

Median, Q1, Q3 and IQR (days)


,Procedure,median_days,q1_days,q3_days,iqr_days
0,Abridged,327.5,243.5,406.0,162.5
1,Non-abridged,359.0,292.0,447.0,155.0



Min / Max (days)


,Procedure,min_days,max_days
0,Abridged,3.0,2388.0
1,Non-abridged,2.0,3284.0


# 7. Relationship between approval activity and disease incidence


In [97]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# Paths (du bist in /src, data ist eine Ebene höher)
# =========================
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "Disease_burden_mapping"

GBD_PATH = DATA_DIR / "Global_disease_burden_statistics_download.csv"
MAP_PATH = DATA_DIR / "Mapping_diseases_disease_classes.csv"

# Approvals dataframe (across all agencies) – muss existieren
APPROVALS_DF = df_overall  # falls anders: hier anpassen

# =========================
# Fixed study years (GBD available up to 2021)
# =========================
START_YEAR = 1995
END_YEAR = 2021

# =========================
# Canonical disease classes (deine Liste)
# =========================
CANONICAL_CLASSES = [
    "Infectious and parasitic diseases",
    "Neoplasms",
    "Diseases of the blood and blood-forming organs",
    "Endocrine, nutritional, and metabolic diseases",
    "Mental and behavioural disorders",
    "Diseases of the nervous system",
    "Diseases of the eye and adnexa",
    "Diseases of the ear and mastoid process",
    "Diseases of the circulatory system",
    "Diseases of the respiratory system",
    "Diseases of the digestive system",
    "Diseases of the skin",
    "Diseases of the musculoskeletal system and connective tissue",
    "Diseases of the genitourinary system",
    "Pregnancy and childbirth",
    "Congenital malformations and chromosomal abnormalities",
    "Injury, poisoning and certain other consequences of external causes",
    "Other",
]

# =========================
# Helpers
# =========================
PLACEHOLDERS = {"not reported", "na", "n/a", "tbd", "none", ""}

def norm_str(s: pd.Series) -> pd.Series:
    return s.astype("string").str.strip()

def approved_mask_from_decision(decision_series: pd.Series) -> pd.Series:
    dec = decision_series.astype("string").str.strip().str.lower().fillna("")
    return dec.str.contains(r"\b(approved|authorised|authorized)\b", regex=True, na=False)

def normalise_disease_class(x) -> str:
    """
    Normalise mapping classes to CANONICAL_CLASSES.
    Fix common typos/variants and map unknowns to 'Other'.
    """
    if pd.isna(x):
        return "Other"

    t = str(x).strip()

    if t.lower() in PLACEHOLDERS:
        return "Other"

    # common typos / variants seen in your message
    fixes = {
        "Diseases of the musculskeletal system and connective tissue":
            "Diseases of the musculoskeletal system and connective tissue",
        "Diseases of the musculskeletal system and connective tissue ":
            "Diseases of the musculoskeletal system and connective tissue",
        "Congenital malformations and chromosal abnormalities":
            "Congenital malformations and chromosomal abnormalities",
        "Injury, poisining and certain other consequences of external causes":
            "Injury, poisoning and certain other consequences of external causes",
        "Injury, poisoning and certain other consequences of external causes ":
            "Injury, poisoning and certain other consequences of external causes",
        "Other ":
            "Other",
    }

    t = fixes.get(t, t)

    # if already canonical -> keep
    if t in CANONICAL_CLASSES:
        return t

    # fallback: try case-insensitive match to canonical list
    lower_map = {c.lower(): c for c in CANONICAL_CLASSES}
    if t.lower() in lower_map:
        return lower_map[t.lower()]

    return "Other"


# =========================
# 1) Load mapping file
# =========================
mapping_raw = pd.read_csv(MAP_PATH, header=None).rename(columns={0: "cause_name"})
class_cols = [c for c in mapping_raw.columns if c != "cause_name"]

mapping_long = (
    mapping_raw
    .melt(id_vars=["cause_name"], value_vars=class_cols, value_name="Disease_class_raw")
    .drop(columns=["variable"])
)

mapping_long["cause_name"] = norm_str(mapping_long["cause_name"])
mapping_long["Disease_class_raw"] = norm_str(mapping_long["Disease_class_raw"])
mapping_long = mapping_long.dropna(subset=["Disease_class_raw"])
mapping_long = mapping_long[~mapping_long["Disease_class_raw"].str.lower().isin(PLACEHOLDERS)]

# normalise to canonical classes (multi-label preserved!)
mapping_long["Disease_class"] = mapping_long["Disease_class_raw"].map(normalise_disease_class)

# drop duplicates
mapping_long = mapping_long.drop_duplicates(subset=["cause_name", "Disease_class"])

# QC: show mapping values that ended up as Other (optional)
qc_other = (
    mapping_long[mapping_long["Disease_class"] == "Other"][["Disease_class_raw"]]
    .drop_duplicates()
    .sort_values("Disease_class_raw")
)
if not qc_other.empty:
    print("\n[QC] Mapping entries that were normalised to 'Other' (check if expected):")
    display(qc_other)


# =========================
# 2) Load GBD file
# =========================
gbd = pd.read_csv(GBD_PATH)

needed_cols = {"measure_name", "metric_name", "year", "cause_name", "val"}
missing = needed_cols - set(gbd.columns)
if missing:
    raise ValueError(f"GBD file missing columns: {missing}. Found: {gbd.columns.tolist()}")

gbd["year"] = pd.to_numeric(gbd["year"], errors="coerce")
gbd["cause_name"] = norm_str(gbd["cause_name"])
gbd["measure_name"] = norm_str(gbd["measure_name"])
gbd["metric_name"] = norm_str(gbd["metric_name"])

MEASURES = ["Incidence", "Prevalence", "Deaths"]
METRIC = "Rate"  # change to "Number" if you want absolute counts

gbd_f = gbd[
    (gbd["measure_name"].isin(MEASURES)) &
    (gbd["metric_name"].str.lower() == METRIC.lower()) &
    (gbd["year"].isin([START_YEAR, END_YEAR]))
].copy()


# =========================
# 3) Map GBD causes -> Disease classes and aggregate
#    (multi-label: causes can contribute to multiple classes)
# =========================
gbd_mapped = gbd_f.merge(
    mapping_long[["cause_name", "Disease_class"]],
    on="cause_name",
    how="left"
)

# Unmapped causes -> 'Other' (so nothing gets dropped silently)
gbd_mapped["Disease_class"] = gbd_mapped["Disease_class"].fillna("Other")
gbd_mapped["Disease_class"] = gbd_mapped["Disease_class"].map(normalise_disease_class)

# Aggregate burden by Disease_class, measure_name, year
burden_class = (
    gbd_mapped.groupby(["Disease_class", "measure_name", "year"], as_index=False)["val"]
    .sum()
)

# Pivot to wide with 1995 & 2021 columns
burden_wide = (
    burden_class.pivot_table(
        index=["Disease_class", "measure_name"],
        columns="year",
        values="val",
        aggfunc="sum"
    )
    .reset_index()
    .rename(columns={
        START_YEAR: f"val_{START_YEAR}",
        END_YEAR: f"val_{END_YEAR}",
    })
)

# % change 1995 -> 2021
burden_wide["pct_change"] = (
    (burden_wide[f"val_{END_YEAR}"] - burden_wide[f"val_{START_YEAR}"]) /
    burden_wide[f"val_{START_YEAR}"] * 100
)

burden_wide["pct_change"] = burden_wide["pct_change"].mask(
    (burden_wide[f"val_{START_YEAR}"] == 0) |
    (burden_wide[f"val_{START_YEAR}"].isna()) |
    (burden_wide[f"val_{END_YEAR}"].isna()),
    np.nan
)

# Wide table with % changes per measure
burden_change_tbl = (
    burden_wide.pivot_table(
        index="Disease_class",
        columns="measure_name",
        values="pct_change",
        aggfunc="first"
    )
    .reindex(CANONICAL_CLASSES)  # keep desired order
    .reset_index()
    .rename(columns={
        "Incidence": f"Incidence_pct_change_{START_YEAR}_{END_YEAR}",
        "Prevalence": f"Prevalence_pct_change_{START_YEAR}_{END_YEAR}",
        "Deaths": f"Deaths_pct_change_{START_YEAR}_{END_YEAR}",
    })
)

# End-year absolute burden (for ranking "high burden")
burden_end_tbl = (
    burden_wide.pivot_table(
        index="Disease_class",
        columns="measure_name",
        values=f"val_{END_YEAR}",
        aggfunc="first"
    )
    .reindex(CANONICAL_CLASSES)
    .reset_index()
    .rename(columns={
        "Incidence": f"Incidence_{END_YEAR}",
        "Prevalence": f"Prevalence_{END_YEAR}",
        "Deaths": f"Deaths_{END_YEAR}",
    })
)

burden_class_summary = burden_change_tbl.merge(burden_end_tbl, on="Disease_class", how="left")


# =========================
# 4) Approvals by Disease class (approved only), per year
# =========================
appr = APPROVALS_DF.copy()
appr = appr[approved_mask_from_decision(appr["Decision"])].copy()

appr["Decision_year"] = pd.to_numeric(appr["Decision_year"], errors="coerce")
appr = appr[appr["Decision_year"].between(START_YEAR, END_YEAR, inclusive="both")]

appr["Disease_class"] = appr["Disease_class(es)"].astype("string").str.split(";")
appr = appr.explode("Disease_class")
appr["Disease_class"] = appr["Disease_class"].astype("string").str.strip()
appr = appr[appr["Disease_class"].notna() & (appr["Disease_class"] != "")]

# Normalise approvals disease classes to canonical list (unknowns -> Other)
appr["Disease_class"] = appr["Disease_class"].map(normalise_disease_class)

appr_counts = (
    appr.groupby(["Disease_class", "Decision_year"], as_index=False)
    .size()
    .rename(columns={"size": "approvals_n"})
)

appr_summary = (
    appr_counts.groupby("Disease_class", as_index=False)
    .agg(
        mean_approvals_per_year=("approvals_n", "mean"),
        max_approvals_per_year=("approvals_n", "max"),
    )
)

peak_year = (
    appr_counts.sort_values(["Disease_class", "approvals_n", "Decision_year"], ascending=[True, False, True])
    .drop_duplicates("Disease_class")[["Disease_class", "Decision_year"]]
    .rename(columns={"Decision_year": "peak_year"})
)

appr_summary = appr_summary.merge(peak_year, on="Disease_class", how="left")

# share (mentions) of approvals by disease class across all approved records
total_mentions = len(appr)
share_tbl = (
    appr["Disease_class"].value_counts()
    .reindex(CANONICAL_CLASSES, fill_value=0)
    .rename_axis("Disease_class")
    .reset_index(name="approvals_mentions_n")
)
share_tbl["approvals_mentions_pct"] = (share_tbl["approvals_mentions_n"] / total_mentions * 100).round(2) if total_mentions else 0.0


# =========================
# 5) Combine burden + approvals
# =========================
combined = (
    burden_class_summary
    .merge(appr_summary, on="Disease_class", how="left")
    .merge(share_tbl[["Disease_class", "approvals_mentions_pct"]], on="Disease_class", how="left")
)

# fill NaNs for classes with no approvals
combined["mean_approvals_per_year"] = combined["mean_approvals_per_year"].fillna(0)
combined["max_approvals_per_year"] = combined["max_approvals_per_year"].fillna(0)
combined["peak_year"] = combined["peak_year"].fillna(pd.NA)

# High burden example: top 10 by Deaths in 2021
high_burden = combined.sort_values(f"Deaths_{END_YEAR}", ascending=False).head(10)

# =========================
# 6) Outputs
# =========================
print("\n=== High-burden classes (top 10 by Deaths in 2021) ===")
display(high_burden[[
    "Disease_class",
    f"Incidence_pct_change_{START_YEAR}_{END_YEAR}",
    f"Prevalence_pct_change_{START_YEAR}_{END_YEAR}",
    f"Deaths_pct_change_{START_YEAR}_{END_YEAR}",
    "mean_approvals_per_year",
    "max_approvals_per_year",
    "peak_year",
    "approvals_mentions_pct",
]])

print("\n=== Combined table (all disease classes; canonical order) ===")
display(combined.set_index("Disease_class").reindex(CANONICAL_CLASSES).reset_index())

print("\n=== Oncology (Neoplasms) quick readout ===")
row = combined.loc[combined["Disease_class"] == "Neoplasms"]
if not row.empty:
    row = row.iloc[0]
    print(f"Neoplasms approvals share (mentions): {row['approvals_mentions_pct']}%")
    print(f"Neoplasms peak approvals/year: {row['max_approvals_per_year']} (peak year: {row['peak_year']})")
else:
    print("No 'Neoplasms' row found (check normalisation).")



[QC] Mapping entries that were normalised to 'Other' (check if expected):


,Disease_class_raw
46,"""Endocrine, nutritional, and metabolic diseases"""
71,"""Injury, poisoning and certain other consequen..."
28,Congenital malformations and chomosomal abnorm...
33,Congenital malformations and chomosomal abnorm...
39,Congenital malformations and chomosomal abnorm...
38,Congenital malformations and chomosomal abnorm...
37,Congenital malformations and chomosomal abnorm...
26,Diseases of the blood and blood-forming organs...
207,Diseases of the blood and blood-forming organs...
140,"Diseases of the circulatory system, Diseases o..."



=== High-burden classes (top 10 by Deaths in 2021) ===


/var/folders/0j/2hyszkt95x38yj11dwpbgx900000gn/T/ipykernel_52530/2393082560.py:57: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return dec.str.contains(r"\b(approved|authorised|authorized)\b", regex=True, na=False)


,Disease_class,Incidence_pct_change_1995_2021,Prevalence_pct_change_1995_2021,Deaths_pct_change_1995_2021,mean_approvals_per_year,max_approvals_per_year,peak_year,approvals_mentions_pct
17,Other,0.395485,-14.310500,-0.529805,3.652174,21,2013.0,0.46
8,Diseases of the circulatory system,30.565718,43.171398,9.566910,75.962963,195,2012.0,11.17
9,Diseases of the respiratory system,-15.609047,-8.750381,-3.325282,49.444444,121,2021.0,7.27
10,Diseases of the digestive system,-10.560121,7.657525,-43.483029,42.962963,94,2017.0,6.32
4,Mental and behavioural disorders,14.380749,7.726185,57.176190,47.962963,108,2018.0,7.05
13,Diseases of the genitourinary system,0.577465,8.243168,70.016911,50.259259,104,2019.0,7.39
1,Neoplasms,12.207185,19.036500,20.453501,61.444444,210,2021.0,9.03
0,Infectious and parasitic diseases,-30.041528,-10.222144,-45.196481,58.962963,136,2021.0,8.67
14,Pregnancy and childbirth,-34.538684,12.706752,-55.201441,10.807692,25,2020.0,1.53
5,Diseases of the nervous system,12.899609,21.112825,16.290626,64.629630,171,2013.0,9.50



=== Combined table (all disease classes; canonical order) ===


,Disease_class,Deaths_pct_change_1995_2021,Incidence_pct_change_1995_2021,Prevalence_pct_change_1995_2021,Deaths_2021,Incidence_2021,Prevalence_2021,mean_approvals_per_year,max_approvals_per_year,peak_year,approvals_mentions_pct
0,Infectious and parasitic diseases,-45.196481,-30.041528,-10.222144,17.010132,4139.579350,4155.939492,58.962963,136,2021.0,8.67
1,Neoplasms,20.453501,12.207185,19.036500,17.276872,585.443357,1266.369029,61.444444,210,2021.0,9.03
2,Diseases of the blood and blood-forming organs,-9.610381,NaN,-5.221601,0.443861,NaN,1482.956246,37.481481,114,2021.0,5.51
3,"Endocrine, nutritional, and metabolic diseases",NaN,NaN,NaN,NaN,NaN,NaN,59.740741,135,2013.0,8.78
4,Mental and behavioural disorders,57.176190,14.380749,7.726185,28.495177,6635.412859,17581.086900,47.962963,108,2018.0,7.05
5,Diseases of the nervous system,16.290626,12.899609,21.112825,9.013901,61.534511,498.492983,64.629630,171,2013.0,9.50
6,Diseases of the eye and adnexa,NaN,NaN,76.437001,NaN,NaN,17108.025123,15.814815,39,2020.0,2.32
7,Diseases of the ear and mastoid process,NaN,NaN,35.023267,NaN,NaN,19587.138279,5.000000,13,2020.0,0.71
8,Diseases of the circulatory system,9.566910,30.565718,43.171398,149.025636,678.513017,7590.450970,75.962963,195,2012.0,11.17
9,Diseases of the respiratory system,-3.325282,-15.609047,-8.750381,55.936940,699.661410,6064.484134,49.444444,121,2021.0,7.27



=== Oncology (Neoplasms) quick readout ===
Neoplasms approvals share (mentions): 9.03%
Neoplasms peak approvals/year: 210 (peak year: 2021.0)


In [93]:
os.getcwd()

'/Users/jacquelinedort/Documents/DrugFork/src'

In [95]:
sorted(gbd["year"].unique())[-10:]

[np.int64(2012),
 np.int64(2013),
 np.int64(2014),
 np.int64(2015),
 np.int64(2016),
 np.int64(2017),
 np.int64(2018),
 np.int64(2019),
 np.int64(2020),
 np.int64(2021)]